<a href="https://colab.research.google.com/github/markajbell/BH/blob/main/GNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
!pip install torch torchvision torchaudio
!pip install torch-geometric  # follow PyG install instructions for your CUDA/CPU
!pip install networkx pandas scikit-learn numpy
import torch
from torch_geometric.data import Data
import pandas as pd
import networkx as nx

# Example inputs
nodes_df = pd.read_csv("nodes.csv")   # id, tier, type, privilege, ...
edges_df = pd.read_csv("edges.csv")   # source, target, weight, relation_type
tier2_to_tier0_df = pd.read_csv("selected_edges_t0.csv")  # tier2_id, tier0_id
nodes_df.head()
# Code to create the 'Tier' column - relocated from cell 7c6653e1
def assign_tier(row):
  if row['Tier0'] == 1:
    return 0
  elif row['Tier1'] == 1:
    return 1
  elif row['Tier2'] == 1:
    return 2
  else:
    return None # Or some other value for nodes that don't fit any tier

# Apply the function AFTER loading the data
nodes_df['Tier'] = nodes_df.apply(assign_tier, axis=1)


# Step 1: Map node IDs to integer indices
node_id_map = {nid: i for i, nid in enumerate(nodes_df['id'])}
num_nodes = len(node_id_map)
node_id_map

# Step 2: Build node features
# One-hot encode tier and type
# Use the already created 'Tier' column
tier_one_hot = pd.get_dummies(nodes_df['Tier'], prefix='Tier')
type_one_hot = pd.get_dummies(nodes_df['type'], prefix='type')

# Add privilege score (normalised)
# Ensure 'weight' column exists before normalizing; assuming 'weight' in nodes_df
if 'weight' in nodes_df.columns:
    nodes_df['privilege_norm'] = nodes_df['weight'] / nodes_df['weight'].max()
else:
    nodes_df['privilege_norm'] = 0.0 # Or handle missing privilege data appropriately


# Combine all features
# Concatenate the one-hot encoded features and the normalized privilege
node_features = pd.concat([tier_one_hot, type_one_hot, nodes_df[['privilege_norm']]], axis=1)

# Convert boolean columns in node_features to integers before creating the tensor
for col in node_features.columns:
    if node_features[col].dtype == 'bool':
        node_features[col] = node_features[col].astype(int)

x = torch.tensor(node_features.values, dtype=torch.float)

# Step 3: Build edge index and edge features
edge_index = torch.tensor(
    [[node_id_map[src], node_id_map[tgt]] for src, tgt in zip(edges_df['source'], edges_df['target'])],
    dtype=torch.long
).t().contiguous()

# Edge features: weight + one-hot relationship type
# Use 'label' column instead of 'relation_type'
relation_one_hot = pd.get_dummies(edges_df['label'], prefix='rel')

# Ensure 'weight' column exists in edges_df
if 'weight' in edges_df.columns:
    edge_features = pd.concat([edges_df[['weight']], relation_one_hot], axis=1)
else:
     edge_features = relation_one_hot # Handle missing weight data appropriately

# Convert boolean columns in edge_features to integers before creating the tensor
for col in edge_features.columns:
    if edge_features[col].dtype == 'bool':
        edge_features[col] = edge_features[col].astype(int)

edge_attr = torch.tensor(edge_features.values, dtype=torch.float)

# Step 4: Build labels for node classification
# Label = 1 if node is Tier 2 with a path to Tier 0
# Assuming 'tier2_id' column exists in tier2_to_tier0_df and contains node IDs
positive_node_ids = tier2_to_tier0_df['source'].tolist() + tier2_to_tier0_df['target'].tolist() # Use source and target from selected edges
positive_node_ids = list(set(positive_node_ids)) # Remove duplicates

labels = torch.zeros(num_nodes, dtype=torch.long)

# Filter for node IDs present in the original nodes_df before mapping
positive_node_ids_in_df = [nid for nid in positive_node_ids if nid in node_id_map]

labels[[node_id_map[nid] for nid in positive_node_ids_in_df]] = 1


# Step 5: Create PyTorch Geometric Data object
data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=labels)

print(data)

In [31]:
"""
adsynth_to_gnn_pipeline.py

End-to-end pipeline to convert ADSynth-derived Active Directory graphs into a format
suitable for Graph Neural Networks (PyTorch Geometric). Also includes a basic GNN
training loop and helper to extract likely/dangerous attack paths using node
risk scores and edge likelihoods.

Author: ChatGPT (for MSc project)
Language: UK English

Dependencies (install in your virtualenv):
    pip install torch torchvision torchaudio
    pip install torch-geometric  # follow PyG install instructions for your CUDA/CPU
    pip install networkx pandas scikit-learn numpy

Notes:
- ADSynth usually outputs nodes and edges (JSON/CSV). This script assumes you
  have two tables/dataframes: `nodes` and `edges`.
- `nodes` should contain at least: id, tier (0/1/2), type (User/Group/Server/OU), privilege
  (numeric score). Additional attributes are supported.
- `edges` should contain at least: source, target, weight (0..1 probability), relation_type.
- The script builds node/edge features, converts to a torch_geometric.data.Data
  object, trains a small GNN to predict whether a Tier 2 node is on a path to Tier 0
  (binary label) and provides helper functions to extract top-k paths ranked by
  combined node-risk and edge-likelihood.

Use this as a template and adapt feature engineering to your specific ADSynth
attributes and research hypotheses.
"""

import math
import json
from typing import List, Tuple, Dict, Any, Optional

import numpy as np
import pandas as pd
import networkx as nx
from sklearn.preprocessing import MinMaxScaler

import torch
from torch import nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv

# ----------------------------- Data conversion ---------------------------------

def load_from_csv(nodes_csv: str, edges_csv: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load nodes and edges CSV files into DataFrames."""
    nodes = pd.read_csv(nodes_csv)
    edges = pd.read_csv(edges_csv)
    return nodes, edges


def load_from_json(nodes_json: str, edges_json: str) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """Load nodes/edges from JSON files (list-of-dicts expected)."""
    with open(nodes_json, 'r') as f:
        nodes = pd.DataFrame(json.load(f))
    with open(edges_json, 'r') as f:
        edges = pd.DataFrame(json.load(f))
    return nodes, edges


def build_node_features(nodes: pd.DataFrame, edges: pd.DataFrame, add_centrality: bool = True) -> Tuple[np.ndarray, List[int]]:
    """
    Build a node feature matrix (numpy) and return an index mapping list
    (node id -> row index). Assumes node id column named 'id'.

    Features included by default:
      - Tier one-hot (tier0, tier1, tier2)
      - Type one-hot (User, Group, Server, OU, other)
      - Normalised privilege score (if privilege column exists)
      - In-degree and out-degree scaled
      - (Optional) betweenness centrality
    """
    # Ensure consistent ordering
    node_ids = list(nodes['id'].astype(str))
    id_to_idx = {nid: i for i, nid in enumerate(node_ids)}

    # Tier one-hot
    tiers = nodes['tier'].astype(int).fillna(-1).astype(int)
    tier_oh = np.zeros((len(nodes), 3), dtype=float)
    for i, t in enumerate(tiers):
        if t in (0, 1, 2):
            tier_oh[i, t] = 1.0

    # Type one-hot
    node_types = nodes['type'].fillna('Unknown').astype(str)
    known_types = ['User', 'Group', 'Server', 'OU']
    type_oh = np.zeros((len(nodes), len(known_types) + 1), dtype=float)
    for i, t in enumerate(node_types):
        if t in known_types:
            type_oh[i, known_types.index(t)] = 1.0
        else:
            type_oh[i, -1] = 1.0

    # Privilege normalised
    if 'privilege' in nodes.columns:
        priv = nodes['privilege'].astype(float).fillna(0.0).values.reshape(-1, 1)
        scaler = MinMaxScaler((0, 1))
        priv_norm = scaler.fit_transform(priv)
    else:
        priv_norm = np.zeros((len(nodes), 1), dtype=float)

    # Build a NetworkX graph to compute degrees / centrality
    G = nx.DiGraph()
    G.add_nodes_from(node_ids)
    for _, row in edges.iterrows():
        src, tgt = str(row['source']), str(row['target'])
        if src in id_to_idx and tgt in id_to_idx:
            G.add_edge(src, tgt, weight=float(row.get('weight', 1.0)))

    indeg = np.array([G.in_degree(n) for n in node_ids], dtype=float).reshape(-1, 1)
    outdeg = np.array([G.out_degree(n) for n in node_ids], dtype=float).reshape(-1, 1)
    # scale degrees
    deg_scale = MaxScaler = MinMaxScaler((0, 1))
    if len(indeg) > 1:
        try:
            indeg = deg_scale.fit_transform(indeg)
            outdeg = deg_scale.transform(outdeg)
        except Exception:
            indeg = indeg / (1.0 + indeg.max())
            outdeg = outdeg / (1.0 + outdeg.max())

    if add_centrality:
        # betweenness centrality (normalised)
        bc = nx.betweenness_centrality(G)
        bc_arr = np.array([bc.get(n, 0.0) for n in node_ids], dtype=float).reshape(-1, 1)
    else:
        bc_arr = np.zeros((len(nodes), 1), dtype=float)

    features = np.hstack([tier_oh, type_oh, priv_norm, indeg, outdeg, bc_arr])
    return features, node_ids


def build_edge_index_and_attr(edges: pd.DataFrame, node_ids: List[str]) -> Tuple[np.ndarray, np.ndarray]:
    """
    Build edge_index (2 x E) and edge_attr matrix.
    Edge attributes: [weight, relation_one_hot...]
    """
    id_to_idx = {nid: i for i, nid in enumerate(node_ids)}

    # relation types
    rels = edges['relation_type'].fillna('unknown').astype(str)
    rel_types = sorted(rels.unique())
    rel_to_idx = {r: i for i, r in enumerate(rel_types)}

    edge_list = []
    edge_attrs = []
    for _, row in edges.iterrows():
        src, tgt = str(row['source']), str(row['target'])
        if src not in id_to_idx or tgt not in id_to_idx:
            continue
        s_idx = id_to_idx[src]
        t_idx = id_to_idx[tgt]
        edge_list.append([s_idx, t_idx])
        weight = float(row.get('weight', 1.0))
        rel_onehot = np.zeros((len(rel_types),), dtype=float)
        rel_onehot[rel_to_idx[row.get('relation_type', 'unknown')]] = 1.0
        edge_attrs.append(np.concatenate(([weight], rel_onehot)))

    edge_index = np.array(edge_list).T  # shape (2, E)
    edge_attr = np.array(edge_attrs)
    return edge_index, edge_attr


# ----------------------------- PyG Data creation --------------------------------

def create_pyg_data(nodes_df: pd.DataFrame, edges_df: pd.DataFrame, positive_tier2_ids: Optional[List[Any]] = None) -> Data:
    """
    Convert nodes/edges dataframes into a torch_geometric.data.Data object.

    positive_tier2_ids: list of node ids (strings/ints) that should be labelled positive (1).
    """
    features, node_id_list = build_node_features(nodes_df, edges_df)
    edge_index, edge_attr = build_edge_index_and_attr(edges_df, node_id_list)

    x = torch.tensor(features, dtype=torch.float)
    edge_index = torch.tensor(edge_index, dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)

    # Build labels: default 0 for all nodes, set 1 for provided positive nodes
    y = torch.zeros(x.size(0), dtype=torch.long)
    if positive_tier2_ids is not None:
        id_to_idx = {nid: i for i, nid in enumerate(node_id_list)}
        for nid in positive_tier2_ids:
            nid_str = str(nid)
            if nid_str in id_to_idx:
                y[id_to_idx[nid_str]] = 1

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    # store mapping for later use
    data.node_id_list = node_id_list
    return data


# ----------------------------- Simple GNN model ---------------------------------

class SimpleGCN(nn.Module):
    def __init__(self, in_channels: int, hidden: int = 64, dropout: float = 0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.lin = nn.Linear(hidden, 2)  # binary classification
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr=None):
        # Note: GCNConv ignores edge_attr. If you wish to use edge attributes,
        # consider NNConv, SAGEConv with edge features, or concatenate and
        # use edge-conditioned methods.
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        out = self.lin(x)
        return out


# ------------------------------ Training utilities -------------------------------

def train_epoch(model: nn.Module, data: Data, optimizer: torch.optim.Optimizer, loss_fn=nn.CrossEntropyLoss()):
    model.train()
    optimizer.zero_grad()
    out = model(data.x, data.edge_index, data.edge_attr)
    # we train on all nodes; for more realistic setup you'd mask training/val/test nodes
    loss = loss_fn(out, data.y)
    loss.backward()
    optimizer.step()
    return loss.item()


def evaluate(model: nn.Module, data: Data) -> Dict[str, float]:
    model.eval()
    with torch.no_grad():
        logits = model(data.x, data.edge_index, data.edge_attr)
        probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()  # probability of class 1
        preds = (probs >= 0.5).astype(int)
        y = data.y.cpu().numpy()

    from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
    metrics = {}
    metrics['precision'] = precision_score(y, preds, zero_division=0)
    metrics['recall'] = recall_score(y, preds, zero_division=0)
    metrics['f1'] = f1_score(y, preds, zero_division=0)
    try:
        metrics['auc'] = roc_auc_score(y, probs)
    except Exception:
        metrics['auc'] = float('nan')
    return metrics, probs


# --------------------------- Path extraction helpers ----------------------------

def extract_topk_attack_paths(edges_df: pd.DataFrame, node_id_list: List[str], node_risk_scores: np.ndarray,
                               source_tier: int = 2, target_tier: int = 0, k: int = 10) -> List[Dict[str, Any]]:
    """
    Given node risk scores (higher = more likely compromised) and edge likelihoods,
    return the top-k paths from any source node in source_tier to any target node in target_tier.

    Path scoring approach (example): For a path p with nodes n0->n1->...->nm and edges e0..e(m-1),
    compute score = product(edge_weight_i * node_risk_score(node_i+1)). To make path-finding easier,
    convert to additive cost: cost = sum(-log(edge_weight) - log(node_risk(node))) and find paths with minimal cost.
    """
    id_to_idx = {nid: i for i, nid in enumerate(node_id_list)}
    G = nx.DiGraph()
    # add nodes with tier info absent here; we'll infer tiers from node ids if needed
    for nid in node_id_list:
        G.add_node(nid)

    for _, r in edges_df.iterrows():
        s, t = str(r['source']), str(r['target'])
        if s not in id_to_idx or t not in id_to_idx:
            continue
        weight = float(r.get('weight', 1.0))
        # avoid zero weights
        weight = max(weight, 1e-6)
        # node risk of target node
        t_idx = id_to_idx[t]
        node_risk = max(float(node_risk_scores[t_idx]), 1e-6)
        cost = -math.log(weight) - math.log(node_risk)
        G.add_edge(s, t, weight=cost, raw_weight=weight)

    # identify source nodes (tier 2) and target nodes (tier 0)
    # For this helper you might supply lists of IDs instead; here we try to parse ids that embed tier info
    src_nodes = [n for n in node_id_list if nodes_id_has_tier(n, source_tier)]
    tgt_nodes = [n for n in node_id_list if nodes_id_has_tier(n, target_tier)]

    # if no tier encoding, user should provide lists; fallback to all nodes
    if len(src_nodes) == 0:
        src_nodes = node_id_list
    if len(tgt_nodes) == 0:
        tgt_nodes = node_id_list

    # compute k-shortest simple paths using Yen's algorithm provided by networkx (if installed)
    all_paths = []
    for s in src_nodes:
        for t in tgt_nodes:
            if s == t or not nx.has_path(G, s, t):
                continue
            try:
                paths = list(nx.shortest_simple_paths(G, s, t, weight='weight'))
            except Exception:
                continue
            for p in paths[:k]:
                # compute raw product score
                prod = 1.0
                for i in range(len(p) - 1):
                    edge_data = G.get_edge_data(p[i], p[i + 1])
                    prod *= edge_data.get('raw_weight', 1.0) * max(node_risk_scores[id_to_idx[p[i + 1]]], 1e-6)
                all_paths.append({'path': p, 'score': prod, 'cost': sum(G[p[i]][p[i+1]]['weight'] for i in range(len(p)-1))})

    # sort by highest score
    all_paths = sorted(all_paths, key=lambda x: x['score'], reverse=True)
    return all_paths[:k]


def nodes_id_has_tier(nid: str, tier: int) -> bool:
    """
    Heuristic: if node id string contains 'T0'/'T1'/'T2' or 'tier0' etc. Adjust for your dataset.
    Replace or extend this in your project to correctly identify tiers from the nodes dataframe.
    """
    s = str(nid).lower()
    return f't{tier}' in s or f'tier{tier}' in s or f'_t{tier}' in s


# ------------------------------- Example usage ---------------------------------

if __name__ == '__main__':
    # Example: Replace these with your actual file paths or DataFrames
    # nodes_df, edges_df = load_from_csv('nodes.csv', 'edges.csv')
    # tier2_to_tier0_df = pd.read_csv('tier2_to_tier0.csv')

    # For demonstration, we'll create a tiny synthetic graph
    nodes_df = pd.DataFrame([
        {'id': 'n1_t2', 'tier': 2, 'type': 'User', 'privilege': 1.0},
        {'id': 'n2_t2', 'tier': 2, 'type': 'Server', 'privilege': 2.0},
        {'id': 'n3_t1', 'tier': 1, 'type': 'Server', 'privilege': 3.0},
        {'id': 'n4_t0', 'tier': 0, 'type': 'Server', 'privilege': 10.0},
    ])

    edges_df = pd.DataFrame([
        {'source': 'n1_t2', 'target': 'n3_t1', 'weight': 0.5, 'relation_type': 'hasSession'},
        {'source': 'n2_t2', 'target': 'n3_t1', 'weight': 0.8, 'relation_type': 'adminTo'},
        {'source': 'n3_t1', 'target': 'n4_t0', 'weight': 0.9, 'relation_type': 'adminTo'},
    ])

    # Suppose tier2_to_tier0_df lists n1_t2 and n2_t2 as having paths to a T0 node
    positive_tier2_ids = ['n1_t2', 'n2_t2']

    data = create_pyg_data(nodes_df, edges_df, positive_tier2_ids=positive_tier2_ids)
    print('Data summary:')
    print(data)

    # Build and train model
    model = SimpleGCN(in_channels=data.x.size(1), hidden=32)
    optim = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    for epoch in range(1, 201):
        loss = train_epoch(model, data, optim)
        if epoch % 50 == 0:
            metrics, probs = evaluate(model, data)
            print(f'Epoch {epoch:03d} loss={loss:.4f} f1={metrics["f1"]:.4f} auc={metrics["auc"]:.4f}')

    metrics, probs = evaluate(model, data)
    print('Final metrics:', metrics)

    # Extract top-k attack paths using predicted node risk (probs) and edge weights from edges_df
    node_risk_scores = probs  # probability each node is 'positive'
    top_paths = extract_topk_attack_paths(edges_df, data.node_id_list, node_risk_scores, source_tier=2, target_tier=0, k=5)
    print('\nTop attack paths:')
    for p in top_paths:
        print(p)

    print('\nPipeline complete. Adapt feature engineering and labelling to your dataset and experiment setup.')


Data summary:
Data(x=[4, 12], edge_index=[2, 3], edge_attr=[3, 3], y=[4], node_id_list=[4])
Epoch 050 loss=0.4261 f1=1.0000 auc=1.0000
Epoch 100 loss=0.4078 f1=1.0000 auc=1.0000
Epoch 150 loss=0.2104 f1=1.0000 auc=1.0000
Epoch 200 loss=0.0773 f1=1.0000 auc=1.0000
Final metrics: {'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'auc': np.float64(1.0)}

Top attack paths:
{'path': ['n2_t2', 'n3_t1', 'n4_t0'], 'score': np.float32(2.4554392e-09), 'cost': 19.824960148003516}
{'path': ['n1_t2', 'n3_t1', 'n4_t0'], 'score': np.float32(1.5346495e-09), 'cost': 20.29496377724925}

Pipeline complete. Adapt feature engineering and labelling to your dataset and experiment setup.


In [32]:
import pandas as pd
import networkx as nx

def attack_graph_stats(edges_df, edges_sub_df, nodes_df=None, nodes_sub_df=None, prob_threshold=0.8):
    # Build NetworkX graphs
    G_full = nx.from_pandas_edgelist(edges_df, 'source', 'target', ['weight', 'pred_prob'], create_using=nx.DiGraph())
    G_sub = nx.from_pandas_edgelist(edges_sub_df, 'source', 'target', ['weight', 'pred_prob'], create_using=nx.DiGraph())

    # Node and edge counts
    full_nodes = G_full.number_of_nodes()
    sub_nodes = G_sub.number_of_nodes()
    full_edges = G_full.number_of_edges()
    sub_edges = G_sub.number_of_edges()

    node_reduction_pct = 100 * (1 - sub_nodes / full_nodes)
    edge_reduction_pct = 100 * (1 - sub_edges / full_edges)

    # Shortest path stats (Tier 2→Tier 0 if nodes_df given)
    avg_shortest_path_full, avg_shortest_path_sub = None, None
    if nodes_df is not None and 'tier' in nodes_df.columns:
        tier2_nodes = nodes_df[nodes_df['tier'] == 2]['id']
        tier0_nodes = nodes_df[nodes_df['tier'] == 0]['id']

        def avg_shortest_path(G, src_nodes, dst_nodes):
            lengths = []
            for s in src_nodes:
                for t in dst_nodes:
                    try:
                        l = nx.shortest_path_length(G, source=s, target=t)
                        lengths.append(l)
                    except nx.NetworkXNoPath:
                        continue
            return sum(lengths) / len(lengths) if lengths else None

        avg_shortest_path_full = avg_shortest_path(G_full, tier2_nodes, tier0_nodes)
        avg_shortest_path_sub = avg_shortest_path(G_sub, tier2_nodes, tier0_nodes)

        # Tier 0 coverage
        tier0_coverage_pct = 100 * (
            len([n for n in tier0_nodes if any(nx.has_path(G_sub, src, n) for src in tier2_nodes)]) /
            len(tier0_nodes)
        )
    else:
        tier0_coverage_pct = None

    # Edge density
    density_full = nx.density(G_full)
    density_sub = nx.density(G_sub)

    # Centrality
    betw_full_max = max(nx.betweenness_centrality(G_full).values())
    betw_sub_max = max(nx.betweenness_centrality(G_sub).values())

    # Prediction confidence stats
    if 'pred_prob' in edges_df.columns and 'pred_prob' in edges_sub_df.columns:
        mean_prob_full = edges_df['pred_prob'].mean()
        mean_prob_sub = edges_sub_df['pred_prob'].mean()

        high_conf_full = (edges_df['pred_prob'] >= prob_threshold).mean() * 100
        high_conf_sub = (edges_sub_df['pred_prob'] >= prob_threshold).mean() * 100
    else:
        mean_prob_full = mean_prob_sub = high_conf_full = high_conf_sub = None

    # Risk-weighted edge fraction
    if 'weight' in edges_df.columns and 'weight' in edges_sub_df.columns:
        high_risk_edges_full = edges_df[edges_df['weight'] >= edges_df['weight'].quantile(0.9)]
        high_risk_edges_sub = edges_sub_df[edges_sub_df['weight'] >= edges_sub_df['weight'].quantile(0.9)]
        high_risk_fraction = 100 * (high_risk_edges_sub.shape[0] / high_risk_edges_full.shape[0])
    else:
        high_risk_fraction = None

    # Assemble results
    stats = {
        'Node Reduction %': node_reduction_pct,
        'Edge Reduction %': edge_reduction_pct,
        'Tier 0 Coverage %': tier0_coverage_pct,
        'Avg Shortest Path (Full)': avg_shortest_path_full,
        'Avg Shortest Path (Sub)': avg_shortest_path_sub,
        'Edge Density (Full)': density_full,
        'Edge Density (Sub)': density_sub,
        'Max Betweenness (Full)': betw_full_max,
        'Max Betweenness (Sub)': betw_sub_max,
        'Mean Pred Prob (Full)': mean_prob_full,
        'Mean Pred Prob (Sub)': mean_prob_sub,
        'High Confidence % (Full)': high_conf_full,
        'High Confidence % (Sub)': high_conf_sub,
        'High Risk Edge Retention %': high_risk_fraction
    }

    return pd.DataFrame([stats])

# Example usage:
# stats_df = attack_graph_stats(edges_df, edges_sub_df, nodes_df, nodes_sub_df)
# print(stats_df.T)  # View as transposed table


In [33]:
"""
compute_subgraph_summary.py

Usage:
  - Prepare CSVs (or adapt to load DataFrames directly):
      nodes_full.csv, edges_full.csv
    Option A (you already have filtered subgraph):
      nodes_sub.csv, edges_sub.csv
    Option B (derive subgraph automatically):
      ensure nodes_full.csv has column 'tier' with values 0/1/2
      the script will compute subgraph = nodes/edges that lie on any path from any Tier-2 to any Tier-0 node.

Outputs:
  - prints a plain-English summary with computed values for the placeholders:
    X, Y, Z, N, R, P, Q, A, B
"""

import pandas as pd
import networkx as nx
import numpy as np
from typing import Optional, Tuple, List, Set

# ---------------- helpers ----------------

def build_graph_from_edges(edges_df: pd.DataFrame, directed: bool = True) -> nx.DiGraph:
    """Create a NetworkX graph from an edges DataFrame. Supports optional attributes 'weight' and 'pred_prob'."""
    if directed:
        G = nx.DiGraph()
    else:
        G = nx.Graph()
    # Ensure columns exist
    edges_cols = edges_df.columns
    for _, row in edges_df.iterrows():
        src = row['source']
        tgt = row['target']
        attrs = {}
        if 'weight' in edges_cols:
            try:
                attrs['weight'] = float(row['weight'])
            except Exception:
                attrs['weight'] = None
        if 'pred_prob' in edges_cols:
            try:
                attrs['pred_prob'] = float(row['pred_prob'])
            except Exception:
                attrs['pred_prob'] = None
        G.add_edge(src, tgt, **attrs)
    return G

def derive_subgraph_by_reachability(nodes_df: pd.DataFrame, edges_df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Build the subgraph containing nodes and edges that lie on any path from any Tier-2 node
    to any Tier-0 node.
    Requires nodes_df to have columns: 'id' and 'tier' (0/1/2).
    """
    G = build_graph_from_edges(edges_df, directed=True)
    nodes_full = nodes_df['id'].astype(str).tolist()
    tier2 = nodes_df[nodes_df['tier'] == 2]['id'].astype(str).tolist()
    tier0 = set(nodes_df[nodes_df['tier'] == 0]['id'].astype(str).tolist())

    # Collect nodes and edges that are on any path from tier2 -> tier0
    nodes_on_paths: Set[str] = set()
    edges_on_paths: Set[Tuple[str, str]] = set()

    for s in tier2:
        for t in tier0:
            if s == t:
                continue
            if not (s in G and t in G):
                continue
            try:
                # Iterate through simple paths (stop reasonably early to avoid explosion)
                # We will use shortest_simple_paths to prioritise shorter paths
                paths = nx.shortest_simple_paths(G, s, t, weight=None)
                # take up to some limit per pair to prevent blow-up
                limit_per_pair = 50
                count = 0
                for p in paths:
                    count += 1
                    for i in range(len(p)):
                        nodes_on_paths.add(p[i])
                        if i < len(p) - 1:
                            edges_on_paths.add((p[i], p[i+1]))
                    if count >= limit_per_pair:
                        break
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                continue

    nodes_sub_df = nodes_df[nodes_df['id'].astype(str).isin(nodes_on_paths)].copy()
    edges_mask = edges_df.apply(lambda r: (str(r['source']), str(r['target'])) in edges_on_paths, axis=1)
    edges_sub_df = edges_df[edges_mask].copy()
    return nodes_sub_df, edges_sub_df

def compute_summary_text(nodes_full: pd.DataFrame,
                         edges_full: pd.DataFrame,
                         nodes_sub: pd.DataFrame,
                         edges_sub: pd.DataFrame,
                         prob_threshold: float = 0.8) -> str:
    """
    Compute metrics and return a plain-English filled summary.
    """
    # Node/edge counts
    V_full = nodes_full['id'].nunique()
    V_sub = nodes_sub['id'].nunique()
    E_full = len(edges_full)
    E_sub = len(edges_sub)

    X_pct = round(100.0 * (1 - V_sub / V_full), 2) if V_full > 0 else None
    Y_pct = round(100.0 * (1 - E_sub / E_full), 2) if E_full > 0 else None

    # Risk sums if 'weight' present
    total_risk_full = edges_full['weight'].sum() if 'weight' in edges_full.columns else None
    total_risk_sub = edges_sub['weight'].sum() if 'weight' in edges_sub.columns else None
    Z_pct = None
    if total_risk_full is not None and total_risk_full > 0:
        Z_pct = round(100.0 * (total_risk_sub / total_risk_full), 2)

    # Shortest path lengths from tier2 to tier0
    # need tiers if present, else compute generic shortest path within subgraph
    if 'tier' in nodes_full.columns:
        tier2 = nodes_full[nodes_full['tier'] == 2]['id'].astype(str).tolist()
        tier0 = nodes_full[nodes_full['tier'] == 0]['id'].astype(str).tolist()
    else:
        # fallback: infer tiers from node ids if possible (not ideal)
        tier2 = nodes_sub[nodes_sub['id'].astype(str).str.contains('2')]['id'].astype(str).tolist()
        tier0 = nodes_sub[nodes_sub['id'].astype(str).str.contains('0')]['id'].astype(str).tolist()

    G_full = build_graph_from_edges(edges_full, directed=True)
    G_sub = build_graph_from_edges(edges_sub, directed=True)

    def avg_shortest_path_between_sets(G: nx.DiGraph, sources: List[str], targets: List[str]) -> Optional[float]:
        lengths = []
        for s in sources:
            for t in targets:
                if s == t or s not in G.nodes() or t not in G.nodes():
                    continue
                try:
                    l = nx.shortest_path_length(G, source=s, target=t)
                    lengths.append(l)
                except (nx.NetworkXNoPath, nx.NodeNotFound):
                    continue
        return float(np.mean(lengths)) if lengths else None

    avg_len_full = avg_shortest_path_between_sets(G_full, tier2, tier0)
    avg_len_sub = avg_shortest_path_between_sets(G_sub, tier2, tier0)

    # For the "N steps" and "R cost" placeholders choose:
    # - N = shortest path length (min over pairs) in subgraph
    # - R = the minimum risk-weighted shortest path cost (if weight exists), else sum of edges of that path
    def min_shortest_and_risk(G: nx.DiGraph, sources: List[str], targets: List[str]) -> Tuple[Optional[int], Optional[float]]:
        min_len = None
        min_risk = None
        for s in sources:
            for t in targets:
                if s == t or s not in G.nodes() or t not in G.nodes():
                    continue
                try:
                    # shortest path by hop-count
                    path = nx.shortest_path(G, source=s, target=t)
                    length = len(path) - 1
                    # compute risk cost if weight exists on edges, else None
                    risk_cost = None
                    if all('weight' in G[u][v] and G[u][v]['weight'] is not None for u, v in zip(path[:-1], path[1:])):
                        risk_cost = sum(float(G[u][v]['weight']) for u, v in zip(path[:-1], path[1:]))
                    if min_len is None or length < min_len:
                        min_len = length
                        min_risk = risk_cost
                    elif length == min_len and risk_cost is not None and min_risk is not None and risk_cost < min_risk:
                        min_risk = risk_cost
                except (nx.NetworkXNoPath, nx.NodeNotFound):
                    continue
        return min_len, min_risk

    N_steps, R_cost = min_shortest_and_risk(G_sub, tier2, tier0)

    # Predicted probability metrics (if pred_prob exists)
    P_mean_full = Q_mean_sub = None
    A_high_full = B_high_sub = None
    if 'pred_prob' in edges_full.columns:
        P_mean_full = float(edges_full['pred_prob'].mean())
        Q_mean_sub = float(edges_sub['pred_prob'].mean())
        A_high_full = round(100.0 * (edges_full['pred_prob'] >= prob_threshold).mean(), 2)
        B_high_sub = round(100.0 * (edges_sub['pred_prob'] >= prob_threshold).mean(), 2)

    # Chokepoints: top k nodes by betweenness in the subgraph
    betw = nx.betweenness_centrality(G_sub)
    top_chokepoints = sorted(betw.items(), key=lambda x: x[1], reverse=True)[:5]
    top_chokepoints_str = ', '.join([f"{n} ({v:.3f})" for n, v in top_chokepoints]) if top_chokepoints else "None"

    # Tier0 coverage (fraction of Tier0 nodes reachable from some Tier2)
    coverage_pct = None
    if tier0:
        reachable_t0 = set()
        for t0 in tier0:
            for t2 in tier2:
                if t0 in G_sub.nodes() and t2 in G_sub.nodes():
                    if nx.has_path(G_sub, t2, t0):
                        reachable_t0.add(t0)
                        break
        coverage_pct = round(100.0 * len(reachable_t0) / len(tier0), 2) if tier0 else None

    # Assemble natural-language summary
    summary = []
    summary.append("Subgraph Summary")
    summary.append(f"After filtering, the reduced subgraph retains {round(100.0 * V_sub / V_full, 2)}% of the original nodes and {round(100.0 * E_sub / E_full, 2)}% of the original edges "
                   f"(node reduction = {X_pct}%, edge reduction = {Y_pct}%).")
    if Z_pct is not None:
        summary.append(f"The subgraph preserves {Z_pct}% of the total risk weight from the full graph (retained risk fraction).")
    else:
        summary.append("No edge 'weight' column present, so risk-weight retention cannot be computed.")

    if N_steps is not None:
        r_cost_text = f" with a combined risk cost of {R_cost}" if R_cost is not None else ""
        summary.append(f"The shortest attack path length from Tier-2 to Tier-0 in the subgraph is {N_steps} step(s){r_cost_text}.")
    else:
        summary.append("No Tier-2 → Tier-0 path exists in the subgraph, so shortest-path statistics are undefined.")

    if P_mean_full is not None:
        summary.append(f"The GNN's mean predicted edge probability across the full graph is {round(P_mean_full, 4)}, "
                       f"and in the subgraph it is {round(Q_mean_sub, 4)}.")
        summary.append(f"The proportion of very high-confidence edges (p ≥ {prob_threshold}) increased from {A_high_full}% to {B_high_sub}% after filtering.")
    else:
        summary.append("No 'pred_prob' column found in edges; prediction-confidence statistics are not available.")

    if coverage_pct is not None:
        summary.append(f"Tier-0 coverage by Tier-2 nodes in the subgraph: {coverage_pct}% of Tier-0 nodes are reachable from at least one Tier-2 node.")
    else:
        summary.append("Tier membership information incomplete; Tier-0 coverage not computed.")

    summary.append(f"Key defensive chokepoints (top nodes by betweenness in the subgraph): {top_chokepoints_str}.")

    return "\n\n".join(summary)


# ---------------- main flow ----------------

if __name__ == "__main__":
    import argparse
    parser = argparse.ArgumentParser(description="Compute plain-English summary for Tier2→Tier0 subgraph.")
    parser.add_argument("--nodes-full", default="nodes_full.csv", help="CSV of full nodes with column 'id' and optionally 'tier'")
    parser.add_argument("--edges-full", default="edges_full.csv", help="CSV of full edges with columns 'source','target', optional 'weight','pred_prob'")
    parser.add_argument("--nodes-sub", default=None, help="CSV of subgraph nodes (optional)")
    parser.add_argument("--edges-sub", default=None, help="CSV of subgraph edges (optional)")
    parser.add_argument("--prob-threshold", type=float, default=0.8, help="threshold for 'very high' confidence edges")
    args = parser.parse_args()

    nodes_full = pd.read_csv(args.nodes_full)
    edges_full = pd.read_csv(args.edges_full)

    if args.edges_sub and args.nodes_sub:
        nodes_sub = pd.read_csv(args.nodes_sub)
        edges_sub = pd.read_csv(args.edges_sub)
    else:
        # derive subgraph automatically
        if 'tier' not in nodes_full.columns:
            raise SystemExit("No nodes_sub provided and 'tier' column missing in nodes_full.csv. Provide nodes_sub/edges_sub or include tier column.")
        nodes_sub, edges_sub = derive_subgraph_by_reachability(nodes_full, edges_full)

    summary_text = compute_summary_text(nodes_full, edges_full, nodes_sub, edges_sub, prob_threshold=args.prob_threshold)
    print("\n" + summary_text + "\n")


usage: colab_kernel_launcher.py [-h] [--nodes-full NODES_FULL]
                                [--edges-full EDGES_FULL]
                                [--nodes-sub NODES_SUB]
                                [--edges-sub EDGES_SUB]
                                [--prob-threshold PROB_THRESHOLD]
colab_kernel_launcher.py: error: unrecognized arguments: -f /root/.local/share/jupyter/runtime/kernel-7712a603-b6db-4e8b-add0-fa7b567a44ce.json


SystemExit: 2

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [18]:
def assign_tier(row):
  if row['Tier0'] == 1:
    return 0
  elif row['Tier1'] == 1:
    return 1
  elif row['Tier2'] == 1:
    return 2
  else:
    return None # Or some other value for nodes that don't fit any tier

nodes_df['Tier'] = nodes_df.apply(assign_tier, axis=1)

# Display the updated DataFrame with the new 'Tier' column
display(nodes_df.head())

,id,properties,type,l1,l2,l3,weight,Tier0,Tier1,Tier2,privilege_norm,Tier
0,6221,"{'domain': 'TESTLAB.LOCALE', 'name': 'TESTLAB....",node,Base,Domain,NaN,1,0,0,0,0.01,NaN
1,6222,"{'domain': 'TESTLAB.LOCALE', 'name': 'Admin@TE...",node,Base,OU,NaN,1,0,0,0,0.01,NaN
2,6223,"{'domain': 'TESTLAB.LOCALE', 'name': 'Tier 1 S...",node,Base,OU,NaN,10,0,1,0,0.10,1.0
3,6224,"{'domain': 'TESTLAB.LOCALE', 'name': 'Tier 2@T...",node,Base,OU,NaN,5,0,0,1,0.05,2.0
4,6225,"{'domain': 'TESTLAB.LOCALE', 'name': 'T0 Admin...",node,Base,OU,NaN,20,1,0,0,0.20,0.0
